# Profiling dos metadados

In [21]:
import sys
import pandas as pd

In [22]:
sys.path.append("..")
from src.transformation.bronze_to_silver import get_df, latest_snapshot
df = get_df(latest_snapshot())

## 1.Tipos

In [12]:
df.dtypes


id                   str
gen                  str
sp                   str
ssp                  str
grp                  str
status               str
en                   str
rec                  str
cnt                  str
loc                  str
lat                  str
lon                  str
alt                  str
type                 str
sex                  str
stage                str
method               str
url                  str
file                 str
file-name            str
sono              object
osci              object
lic                  str
q                    str
length               str
time                 str
date                 str
uploaded             str
also              object
rmk                  str
animal-seen          str
playback-used        str
temp              object
regnr                str
auto                 str
dvc                  str
mic                  str
smp                  str
annotation-set    object
_meta             object


Quase tudo veio como str, os restantes vieram como dicionários aninhados.

## 2. Nulos e Vazios

In [16]:
df.isna().sum()

id                    0
gen                   0
sp                    0
ssp                   0
grp                   0
status                0
en                    0
rec                   0
cnt                   0
loc                  32
lat                1841
lon                1837
alt                   0
type                  0
sex                   0
stage                 0
method                0
url                   0
file                 32
file-name            32
sono                  0
osci                  0
lic                   0
q                     0
length                0
time                  0
date                  0
uploaded              0
also                  0
rmk                   0
animal-seen           0
playback-used         0
temp              60552
regnr                 0
auto                  0
dvc                   0
mic                   0
smp                  14
annotation-set        0
_meta             60520
dtype: int64

In [17]:
(df == "").sum()

id                    0
gen                   0
sp                    0
ssp               51356
grp                   0
status                0
en                    0
rec                   0
cnt                   0
loc                   3
lat                   0
lon                   0
alt                2712
type                209
sex               41573
stage             44676
method               14
url                   0
file                  0
file-name             0
sono                  0
osci                  0
lic                   0
q                     0
length                0
time                  1
date                  0
uploaded              0
also                  0
rmk               32610
animal-seen           0
playback-used         0
temp                  0
regnr             60527
auto                  0
dvc               48460
mic               49033
smp                   0
annotation-set        0
_meta                 0
dtype: int64

In [18]:
(df["also"].str.len() == 0).sum()

np.int64(42173)

In [19]:
(df["annotation-set"].str.len() == 0).sum()

np.int64(60541)

~ 1.8k coordenadas nulas;
temp nulo pra todos dados;
32 gravações de espécies restritas;
~ 51k ssp vazias;
~ 41k sex vazias;
~ 44k stage vazias;
~ 32k rmk vazias;
regnr vazia pra todas as linhas;
~ 48k dvc vazias;
~ 49k mic vazias;
somente 11 anotações não vazias;
~ 42k de gravações sem espécie de fundo;

## 3. Duplicados

In [15]:
df["id"].duplicated().sum()

np.int64(0)

Nenhum, como esperado

## 4. Exploração

In [25]:
df["smp"].value_counts(dropna=False)

smp
44100     42923
48000     15668
22050      1022
32000       412
16000       250
24000       142
96000        89
11025        19
NaN          14
8000         10
214285        3
Name: count, dtype: int64

In [27]:
df["q"].value_counts()

q
B    32819
A    27733
Name: count, dtype: int64

In [28]:
df["type"].value_counts(dropna=False)

type
song                                                                          30508
call                                                                          15638
call, song                                                                     4407
flight call                                                                    1363
canto                                                                          1076
                                                                              ...  
duet, duet chatter                                                                1
call, flight call, call perched birds, and communal flight call at the end        1
call, duet, flight call, interaction calls by flock                               1
territorial call                                                                  1
unknown                                                                           1
Name: count, Length: 1145, dtype: int64

In [29]:
df["method"].value_counts(dropna=False)

method
field recording                 54907
in the hand                      5622
                                   14
field recording, in the hand        6
unknown                             2
in net                              1
Name: count, dtype: int64

In [30]:
df["status"].value_counts(dropna=False)

status
identified      60160
unidentified      268
questioned        121
unconfirmed         3
Name: count, dtype: int64

In [31]:
df["length"].str.count(":").value_counts(dropna=False)

length
1    60549
2        3
Name: count, dtype: int64

- Taxa de amostragem mais comum: 44100
- Todas gravações são de qualidade A e B
- Maioria do tipo call e song
- Maioria é gravação de campo
- 60k são identificadas
- 60549 gravações de minutos, 3 com horas

## Decisões finais

### Mantidos inalterados
- rec, file, lic, status, q

### Tipos
- id -> int
- lat, lon -> float
- smp -> Int64

### Transformações
- gen, sp -> virarão coluna única 'species'
- length -> segundos, Int64
- auto, dvc, mic, method, type -> Tratamento nulos
- also: vira tabela separada, 'background_species'

### Remoções
- gen, sp -> virarão species
- grp, cnt -> Sempre constantes
- length -> vira length_s
- ssp, sex, stage, regnr, temp, rmk -> Muitos nulos e pouco relevantes
- loc, alt, en, animal-seen, playback-used, time, date, uploaded, url, file-name, _meta -> pouco relevantes
- sono, osci, annotation-set -> listas/dicts e pouco relevantes

